<a href="https://colab.research.google.com/github/SabareeshSS/April/blob/master_nb/FineTuneLlama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
from google.colab import files

files.download("criticsab.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
import shutil
shutil.make_archive("criticsab","zip","criticsab")

'/content/criticsab.zip'

In [22]:
model.save_pretrained("criticsab")
tokenizer.save_pretrained("criticsab")

('criticsab/tokenizer_config.json',
 'criticsab/special_tokens_map.json',
 'criticsab/tokenizer.json')

In [21]:
FastLanguageModel.for_inference(model)

inputs = tokenizer( [alpaca_prompt_ip.format("Analyze","","")],return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appopriately completes the request.\n\n###Instruction:\nAnalyze\n\n###Input:\n\n\n###Response:\n### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ### ###']

In [17]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 125 | Num Epochs = 4 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)


Step,Training Loss
1,0.000400
2,0.000300
3,0.000100
4,0.000000
5,0.000100
6,0.000100
7,0.000000
8,0.000000
9,0.000000
10,0.000000


In [16]:
from trl import SFTTrainer

from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer (
                      model = model,
                      tokenizer = tokenizer,
                      train_dataset = dataset,
                      dataset_text_field="text",
                      max_seq_length = max_seq_length,
                      dataset_num_proc = 2,
                      packing = False,
                      args = TrainingArguments(
                          per_device_train_batch_size=2,
                          gradient_accumulation_steps=4 ,
                          warmup_steps = 5,
                          max_steps=60,
                          learning_rate = 2e-4,
                          fp16= not is_bfloat16_supported(),
                          bf16=is_bfloat16_supported(),
                          logging_steps=1,
                          optim="adamw_8bit",
                          weight_decay=0.01,
                          lr_scheduler_type="linear",
                          seed=3407,
                          output_dir = "outputs"
                      ),
                      )

Map (num_proc=2):   0%|          | 0/125 [00:00<?, ? examples/s]

In [15]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="/customdataset3.csv", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

In [7]:
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
  instructions = examples["Instruction"]
  inputs = examples["Input"]
  outputs = examples["Response"]

  texts = []
  for instruction, input, output in zip(instructions, inputs, outputs):
    text = alpaca_prompt_ip.format(instruction, input, output) + EOS_TOKEN
    texts.append(text)
  return {"text": texts}

In [6]:
alpaca_prompt_ip = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appopriately completes the request.

###Instruction:
{}

###Input:
{}

###Response:
{}"""

In [5]:
model = FastLanguageModel.get_peft_model (
          model,
          target_modules=["q_proj", "k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj",],
          lora_alpha = 16,
          lora_dropout = 0,
          bias = "none",
          use_gradient_checkpointing = "unsloth",
          random_state = 3407,
          use_rslora = False,
          loftq_config = None
                   )

Unsloth 2025.5.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length=2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

Unsloth: Patching Xformers to fix some performance issues.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.3.0+cu121 with CUDA 1201 (you have 2.6.0+cu124)
    Python  3.11.9 (you have 3.11.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.5.4: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [3]:
pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 23.1 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 0.15.2
    Uninstalling trl-0.15.2:
      Successfully uninstalled trl-0.15.2


In [2]:
pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-0pmx2n8_/unsloth_34dfae7667ef46089d8cc5f7a5bd3318
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-0pmx2n8_/unsloth_34dfae7667ef46089d8cc5f7a5bd3318
  Resolved https://github.com/unslothai/unsloth.git to commit 6e771a0a24998614494d1cf11429e1d29476c114
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
